# Demo API Ollama/ChatGPT

In [1]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer, AutoModel, pipeline
from gensim.models import Word2Vec
import numpy as np

## Using local models (Ollama)

::: callout-note
### 1. Install Ollama
Ollama is a tool that allows you to run language models locally on your machine.  
It is available for macOS, Linux, and Windows (via WSL).  
👉 [Download Ollama](https://ollama.com/download)

:::

::: callout-note
### 2. Download a model
Ollama offers several models (LLaMA, Gemma, Mistral, etc.). Here we download and load the gemma3:4b model.
To download one, use the command:

`ollama pull gemma3:4b`


This downloads the `gemma3:4b` model.
You can then view the models available locally with:

`ollama list`

::: callout-note
### 3. Run a simple query
Once the model is installed, you can test it directly in the terminal:

`ollama run gemma3:4b`

::: callout-note
### 4. Using Ollama in Python
Ollama provides a local API that can be queried via the `ollama` package for Python.  
Here is a minimal example of sentiment analysis:

In [11]:
import ollama
import json

# 1. The input text (English translation of a French customer complaint)
text = "I can't change my vehicle registration address online, and the local government won't listen because it's no longer within their jurisdiction. It's been four months now."

# 2. The Prompt
# We instruct the model to act as a sentiment analyzer and enforce JSON output.
prompt = f"""
You are a sentiment analysis expert.
Analyze the sentiment of the following text.
Classify it as: Positive, Negative, or Neutral.

Return ONLY a JSON object with the key "label".
Example: {{"label": "Negative"}}

Text to analyze: "{text}"
"""

try:
    # 3. The API Call
    response = ollama.chat(
        model="gemma3:4b",  # Ensure this model is installed via `ollama pull gemma:2b`
        messages=[{"role": "user", "content": prompt}],
        format="json"      # Crucial for easy parsing
    )
    
    # 4. Parsing the result
    result = json.loads(response["message"]["content"])
    
    print(f"Text: {text}")
    print(f"Sentiment: {result.get('label')}")

except Exception as e:
    print(f"Error: {e}")
    print("Tip: Ensure Ollama is running in your terminal.")

Text: I can't change my vehicle registration address online, and the local government won't listen because it's no longer within their jurisdiction. It's been four months now.
Sentiment: Negative


::: {.callout-tip title="Exhaustive List of Ollama Generation Parameters"}
For total control over Ollama model outputs, you can specify a series of parameters via the `options` object. These settings allow you to adjust everything from creativity to response structure. Here is the complete list of available options:

### Creativity Control (Sampling)

-   **`temperature`** (e.g., `0.7`)
    Adjusts the level of randomness. A value close to `0` makes the model deterministic and factual, while a high value (`>1.0`) makes it more creative, or even chaotic. *Default: 0.8*.

-   **`top_k`** (e.g., `40`)
    Filters the vocabulary to the `k` most probable words before each new generation step. Prevents the model from choosing overly bizarre words. *Default: 40*.

-   **`top_p`** (e.g., `0.9`)
    Filters vocabulary by selecting the smallest set of words whose cumulative probability reaches the threshold `p`. A dynamic alternative to `top_k`. It is recommended to use only one of the two. *Default: 0.9*.

-   **`seed`** (e.g., `42`)
    Sets a "seed" for random generation. By using the same seed for the same prompt, you will consistently get the same response, which is useful for reproducibility. *Default: 0 (random)*.

-   **`tfs_z`** (e.g., `1`)
    Enables *tail free sampling*. This technique removes words considered unlikely from the "tail" of the probability distribution. A value of `1` disables this filter. *Default: 1*.

### Repetition Control

-   **`repeat_penalty`** (e.g., `1.1`)
    Penalizes words that have already been used in the response, thereby reducing repetitions. A value greater than `1` increases the penalty. *Default: 1.1*.

-   **`repeat_last_n`** (e.g., `64`)
    Sets how many previous words the repetition penalty applies to. A value of `-1` applies it to the entire context window. *Default: 64*.

### Length and Stop Control

-   **`num_predict`** (e.g., `128`)
    The maximum number of words (tokens) to generate. A value of `-1` means unlimited generation until a natural stop or the end of the context. *Default: 128*.

-   **`stop`** (e.g., `["\n", "User:"]`)
    A list of strings that, if generated, will immediately stop the response. Useful for controlling output format (e.g., stopping after a line break).

### Context Control

-   **`num_ctx`** (e.g., `2048`)
    Sets the size of the context window (in tokens) that the model will take into account to generate its response. *Default: 2048*.

### Advanced Sampling Methods (Mirostat)

Mirostat is an algorithm that dynamically adjusts the surprise rate (perplexity) of responses to maintain a certain coherence.

-   **`mirostat`** (e.g., `1`)
    Enables Mirostat. `0` = disabled, `1` = Mirostat v1, `2` = Mirostat v2. *Default: 0*.

-   **`mirostat_tau`** (e.g., `5.0`)
    The target "surprise rate". Influences text coherence. Lower values yield more coherent text. *Default: 5.0*.

-   **`mirostat_eta`** (e.g., `0.1`)
    The "learning rate". Determines how quickly the algorithm adjusts to reach the target `surprise rate`. *Default: 0.1*.

:::

```python
# Example code combining multiple parameters for a specific task
response = ollama.chat(
    model="gemma3:4b",
    messages=[{'role': 'user', 'content': 'Tell a very short story.'}],
    options={
        'seed': 101,
        'temperature': 0.9,
        'top_k': 50,
        'repeat_penalty': 1.2,
        'num_predict': 100,
        'stop': ['.'] 
    }
)
print(response['message']['content'])

In [9]:
# Example of using options for a more factual response
response = ollama.chat(
    model="gemma3:4b",
    messages=[{'role': 'user', 'content': 'Explain what the 4Ps are in marketing. Elaborate well.'}],
    options={
        'temperature': 0.3,
        'top_k': 20
    }
)

print(response['message']['content'])

Okay, let's break down the 4Ps of marketing – a foundational concept that’s still incredibly relevant today. They represent the core elements a business needs to consider when developing and executing a marketing strategy. Here’s a detailed explanation of each:

**The 4Ps of Marketing:**

1. **Product:** This is the core offering – what you're actually selling. It’s *much* more than just the physical item. It encompasses everything about what the customer is buying.

   * **What it includes:**
      * **Features:** The specific characteristics and functionalities of the product. (e.g., a smartphone’s camera resolution, battery life, operating system)
      * **Benefits:** What the customer *gets* from using the product. (e.g., a smartphone allows you to take high-quality photos, stay connected, and access information)
      * **Quality:** The level of performance, durability, and reliability.
      * **Design:** The aesthetic appeal and usability of the product.
      * **Brand Name & 

## Open dataset /data/corpus_cleaned_ner.csv

In [35]:
data = pd.read_csv('data/corpus_cleaned_ner.csv')

In [38]:
data.head()

,id,artist,title,year,album,lyrics,n_words,pageviews,contributors,url,marque,produit,emotion,lieu,artiste,pays,ville,date,region
0,1,Dry,94310,2012.0,Tôt ou tard,"Un balle dans la tête, je te la place comme Be...",748,0,8,https://genius.com/Dry-94310-lyrics,NaN,NaN,NaN,NaN,NaN,NaN,Amsterdam,"9.4.310, 9.4.310, 9.4.310, 9.4.310",NaN
1,2,Mafia K’1 Fry,Au bon vieux temps,2007.0,Jusqu’à la mort,"Et je les entends tous les Zoulous, ils parlen...",958,7514,8,https://genius.com/Mafia-k1-fry-au-bon-vieux-t...,NaN,NaN,NaN,NaN,"K'1 Fry, K'1 Fry",NaN,NaN,NaN,NaN
2,3,DJ Hamida,Attrape-Moi Si Tu Peux,NaN,Mix Party 2015,"Oui je sais qu’t’as envie\nOn cherche, on se d...",367,0,4,https://genius.com/Dj-hamida-attrape-moi-si-tu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Kery James,94 c’est le Barça Remix,NaN,NaN,"C'est pour les rudeboys, c'est pour les caille...",535,0,7,https://genius.com/Kery-james-94-cest-le-barca...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Dry,14 ans déjà,2013.0,Maintenant ou jamais,"Tu sais, j'crois qu'j'ai pas réalisé tout de s...",653,5037,8,https://genius.com/Dry-14-ans-deja-lyrics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


data = pd.read_csv('/data/corpus_cleaned_ner.csv')

## Brand detection in rap lyrics with ChatGPT




In [23]:
import os
import json
import pandas as pd
from openai import OpenAI

# Assure-toi que ta clé est bien chargée
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

lyrics = [
    "J'remplis la valise, vise la maille, pas les likes, Gucci sur le hoodie",
    "Pas de marque, juste des rêves et des nuits blanches",
    "Prada sur les baskets, Audi sur le périph, pas de sentiments",
    "On encaisse en silence, Nike et Adidas dans le sac de sport",
]

df = pd.DataFrame({"lyric": lyrics})

# Note: En mode JSON, il faut demander un objet {}, pas juste une liste [].
SYSTEM = (
    "You are a brand-spotter for rap lyrics. "
    "Given a lyric, return a JSON object with a key 'brands' containing the list of explicit brand names. "
    "Example: {\"brands\": [\"Nike\"]}. If none, return {\"brands\": []}."
)

def detect_brands(text: str):
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",  # Utilise un modèle réel et rapide
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"Lyric: {text}"},
            ],
            temperature=0,  # Température à 0 pour la rigueur
            response_format={"type": "json_object"} # Force le format JSON valide
        )
        
        raw = resp.choices[0].message.content
        
        # Parsing sécurisé
        data = json.loads(raw)
        return data.get("brands", [])
        
    except Exception as e:
        print(f"Erreur sur le texte '{text}': {e}")
        return []

# Exécution
df["brands"] = df["lyric"].apply(detect_brands)

# Affichage propre
print(df)

                                               lyric          brands
0  J'remplis la valise, vise la maille, pas les l...         [Gucci]
1  Pas de marque, juste des rêves et des nuits bl...              []
2  Prada sur les baskets, Audi sur le périph, pas...   [Prada, Audi]
3  On encaisse en silence, Nike et Adidas dans le...  [Nike, Adidas]


In [27]:
import os
import json
import pandas as pd
from openai import OpenAI

# Chargement de la clé API
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("La clé OPENAI_API_KEY n'est pas définie.")

client = OpenAI(api_key=api_key)

# Données
lyrics = [
    "J'remplis la valise, vise la maille, pas les likes, Gucci sur le hoodie",
    "Pas de marque, juste des rêves et des nuits blanches",
    "Prada sur les baskets, Audi sur le périph, pas de sentiments",
    "On encaisse en silence, Nike et Adidas dans le sac de sport",
]

df = pd.DataFrame({"lyric": lyrics})

# Prompt système (Mode JSON Object)
SYSTEM = (
    "You are a brand-spotter for rap lyrics. "
    "Given a lyric, return a JSON object with a key 'brands' containing the list of explicit brand names. "
    "Example: {\"brands\": [\"Nike\"]}. If none, return {\"brands\": []}."
)

def detect_brands(text: str):
    try:
        resp = client.chat.completions.create(
            model="gpt-5-nano-2025-08-07",  # Modèle spécifique demandé
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"Lyric: {text}"},
            ],
            # Suppression de temperature et max_completion_tokens comme demandé
            response_format={"type": "json_object"} 
        )
        
        raw = resp.choices[0].message.content
        
        # Parsing JSON
        data = json.loads(raw)
        return data.get("brands", [])
        
    except Exception as e:
        print(f"Erreur sur le texte '{text}': {e}")
        return []

# Exécution
df["brands"] = df["lyric"].apply(detect_brands)

# Affichage
print(df)

                                               lyric          brands
0  J'remplis la valise, vise la maille, pas les l...         [Gucci]
1  Pas de marque, juste des rêves et des nuits bl...              []
2  Prada sur les baskets, Audi sur le périph, pas...   [Prada, Audi]
3  On encaisse en silence, Nike et Adidas dans le...  [Nike, Adidas]


## Structure With Pydantic

In [28]:
import os
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# 1. On définit la forme exacte de la réponse attendue
class BrandResponse(BaseModel):
    brands: list[str]

lyrics = [
    "J'remplis la valise, vise la maille, pas les likes, Gucci sur le hoodie",
    "Pas de marque, juste des rêves et des nuits blanches",
    "Prada sur les baskets, Audi sur le périph, pas de sentiments",
    "On encaisse en silence, Nike et Adidas dans le sac de sport",
]

df = pd.DataFrame({"lyric": lyrics})

def detect_brands(text: str):
    try:
        # On utilise .parse() au lieu de .create()
        completion = client.beta.chat.completions.parse(
            model="gpt-5-nano-2025-08-07",
            messages=[
                {"role": "system", "content": "Extract all explicit brand names."},
                {"role": "user", "content": text},
            ],
            response_format=BrandResponse, # <--- La magie est ici
        )
        # On récupère l'objet Python direct (pas de parsing manuel)
        return completion.choices[0].message.parsed.brands
    except Exception as e:
        print(f"Erreur : {e}")
        return []

df["brands"] = df["lyric"].apply(detect_brands)
print(df)

                                               lyric          brands
0  J'remplis la valise, vise la maille, pas les l...         [Gucci]
1  Pas de marque, juste des rêves et des nuits bl...              []
2  Prada sur les baskets, Audi sur le périph, pas...   [Prada, Audi]
3  On encaisse en silence, Nike et Adidas dans le...  [Nike, Adidas]


## With Instructor

In [30]:
#!pip instal instructor
import os
import pandas as pd
import instructor
from openai import OpenAI
from pydantic import BaseModel

# On "patch" le client pour lui ajouter les super-pouvoirs
client = instructor.from_openai(OpenAI(api_key=os.getenv("OPENAI_API_KEY")))

class BrandResponse(BaseModel):
    brands: list[str]

lyrics = [
    "J'remplis la valise, vise la maille, pas les likes, Gucci sur le hoodie",
    "Pas de marque, juste des rêves et des nuits blanches",
    "Prada sur les baskets, Audi sur le périph, pas de sentiments",
    "On encaisse en silence, Nike et Adidas dans le sac de sport",
]

df = pd.DataFrame({"lyric": lyrics})

def detect_brands(text: str):
    try:
        # response_model fait tout le travail (parsing + validation)
        resp = client.chat.completions.create(
            model="gpt-5-nano-2025-08-07",
            response_model=BrandResponse, 
            messages=[
                {"role": "system", "content": "Extract brand names."},
                {"role": "user", "content": text},
            ],
        )
        return resp.brands
    except Exception as e:
        print(f"Erreur : {e}")
        return []

df["brands"] = df["lyric"].apply(detect_brands)
print(df)

                                               lyric          brands
0  J'remplis la valise, vise la maille, pas les l...         [Gucci]
1  Pas de marque, juste des rêves et des nuits bl...              []
2  Prada sur les baskets, Audi sur le périph, pas...   [Prada, Audi]
3  On encaisse en silence, Nike et Adidas dans le...  [Nike, Adidas]


# Methodology: Structured Extraction with Instructor & Pydantic

Instead of using standard Regular Expressions or fragile JSON parsing (`json.loads`), this workflow uses the **Instructor** library combined with **Pydantic**. This is currently the industry standard for reliable data extraction from LLMs.

### The Stack

1.  **Pydantic (`BaseModel`) → The Blueprint**
    * We use Pydantic to define the **exact structure** of the data we want.
    * It acts as a strict contract (e.g., *"I want a list of strings, not a raw sentence"*).
    * *Role: Validation & Type Safety.*

2.  **Instructor → The Enforcer**
    * Instructor wraps the OpenAI client to handle the complexity of **Function Calling**.
    * It converts our Pydantic class into a JSON Schema that the LLM understands.
    * **The "Retry Loop"**: If the LLM returns invalid data (e.g., a string instead of a list), Instructor automatically catches the error and asks the LLM to correct itself before returning the result to us.
    * *Role: API Communication & Error Handling.*

### How it works (The Flow)

> **Code**: `class Brand(BaseModel): ...`

⬇️

> **Instructor**: Translates Python class to JSON Schema for the LLM.

⬇️

> **LLM**: Generates structured JSON filling the schema.

⬇️

> **Instructor**: Validates data. (❌ If error: Automatic Retry / ✅ If success: Returns Object).

⬇️

> **Result**: A clean Python object (no parsing required).

In [40]:
import os, pandas as pd, instructor
from openai import OpenAI
from pydantic import BaseModel

# 1. Setup
client = instructor.from_openai(OpenAI(api_key=os.getenv("OPENAI_API_KEY")))
class BrandList(BaseModel): brands: list[str]
output_csv = 'rap_brands.csv'

# 2. Load & Sample 20 rows
df = pd.read_csv('data/corpus_cleaned_ner.csv').sample(20)

# 3. Create CSV header if it doesn't exist
if not os.path.exists(output_csv):
    pd.DataFrame(columns=['id', 'artist', 'title', 'brands']).to_csv(output_csv, index=False)

# 4. Loop & Save (Append mode)
for i, row in df.iterrows():
    if pd.isna(row['lyrics']): continue # Skip empty lyrics

    # Extract
    resp = client.chat.completions.create(
        model="gpt-5-nano-2025-08-07",
        response_model=BrandList,
        messages=[{"role": "user", "content": f"Extract explicit brand names: {row['lyrics'][:3000]}"}]
    )

    # Save immediately
    print(f"Saved: {row['title']} -> {resp.brands}")
    pd.DataFrame({
        'id': [row['id']], 
        'artist': [row['artist']], 
        'title': [row['title']], 
        'brands': [", ".join(resp.brands)]
    }).to_csv(output_csv, mode='a', header=False, index=False)

Saved: Sierra -> ['Glock', 'Saphir']
Saved: ​eritriste -> ['Aldi', 'Lidl', 'Leader Price', 'OnlyFans']
Saved: Qui on est -> ['Blunt', 'Pirelli', 'Juventus']
Saved: Pull Up -> ['Skuna', 'Sad', 'Jordan', 'Heetch']
Saved: Rap Fighter Cup #3 - Nidpool (Faf Larage) VS Mytoman (Greg Frite) -> ['Winamp', 'Lexomyl', 'Valium', 'Boot Camp', 'Chimichanga', 'Rap Fighter']
Saved: Fais la Passe -> ['Fight Club']
Saved: Ça c’est champion ! / So Good New -> ['PTT', 'Birdy Nam Nam', 'Stade 2', 'P. Diddy', 'Sumotoris', 'Dr Vince', 'Gérard Baste']
Saved: Le prix de la réussite -> ['2K', 'Richard Mille']
Saved: Baptême de plongée -> []
Saved: FAIS NOUS VOIR -> ['Burberry', 'Veuve Clicquot', 'Audi']
Saved: ​​​​la grande désillusion -> []
Saved: Freestyle Rapunchline (RP House #8) -> ['Elouan Raptors', 'Brr', 'La fusée']
Saved: Happy Face -> ['Palace Prod']
Saved: Back -> ["Jack Daniel's", 'Pokemon', 'Secret Story']
Saved: YES BABE (Le Sky #7) -> ['Burberry', 'Patek Phillip']
Saved: Le coffre-fort ne suivra

In [42]:
import os
import pandas as pd
import instructor
from pydantic import BaseModel, Field

# 1. Setup: Initialize Instructor via the provider
# Automatically fetches OPENAI_API_KEY from environment variables
client = instructor.from_provider("openai/gpt-4o-mini")

# 2. Define the Target Structure
class BrandResponse(BaseModel):
    # Field description acts as a prompt to guide the LLM
    brands: list[str] = Field(description="List of explicit commercial brands mentioned (e.g., Nike, Audi).")

# 3. File Preparation
output_csv = 'rap_brands_v2.csv'
# Load 20 random rows for testing
df = pd.read_csv('data/corpus_cleaned_ner.csv').sample(20)

# Initialize CSV with headers if the file does not exist
if not os.path.exists(output_csv):
    pd.DataFrame(columns=['id', 'artist', 'title', 'brands']).to_csv(output_csv, index=False)

# 4. Main Extraction Loop
for i, row in df.iterrows():
    if pd.isna(row['lyrics']): continue

    try:
        # Perform extraction (Handles validation and retries automatically)
        resp = client.chat.completions.create(
            response_model=BrandResponse,
            messages=[
                {"role": "system", "content": "You are a precise data extraction assistant."},
                {"role": "user", "content": f"Extract brand names from this text: {row['lyrics'][:3000]}"}
            ],
            max_retries=3  # Automatically retry up to 3 times on validation errors
        )
        
        brands_str = ", ".join(resp.brands)
        print(f"✅ {row['title']} -> [{brands_str}]")

        # Save immediately to disk (Append mode)
        pd.DataFrame({
            'id': [row['id']], 
            'artist': [row['artist']], 
            'title': [row['title']], 
            'brands': [brands_str]
        }).to_csv(output_csv, mode='a', header=False, index=False)

    except Exception as e:
        print(f"❌ Failed on {row['title']}: {e}")

✅ Jacquemüs -> [Jacquemüs, Helmut Lang, Rick Owens]
✅ De temps en temps -> [EasyJet, Kouchner, Catherine Zeta-Jones, Luna Park, L'État, Nakk Mendosa]
✅ Free Gouap 5 -> [Gouap, Maz, kush]
✅ La Vie Ou La Mort (Remix) -> [Kellogg's, Bang Bros, Juventus, Arma Jacks, Hefty Blacks, Dolly]
✅ Nos rancunes -> [Gomez Addams]
✅ Long Beach -> [Play, Long Beach, Saint James, Caresse Antillaise, Waze]
✅ Bouton rouge -> [Furlax]
✅ Simba II* -> [CR, V, C, P, OG Kush, Miami, DD, Meruem, N'DA, Essonne, The World, L.A, Itachi, Bercy, 185, Z double O]
✅ Dernière page -> [Egaré]
✅ Freestyle Couvre feu OKLMRadio -> [Neman, zeeman, ozimen, OCB, Mala, arabica, gamos, Hess]
✅ Mélodrame -> [Benz]
✅ Tupac -> [Pierre Woodman]
✅ Fala diss -> [Neo, Legolas, LUTĒCE]
✅ Bill Gates -> [Givenchy, Berlusconi, Bill, Kim, Milito, Phillip Mo', OVG, Larry, Tariq, Osirus, Jacky, Milka, Wiz K, Kim K, NASA, Kixou, BPK, Heenok, CIA, Actavis, Nascar, Fredo]
✅ Trouble -> []
✅ Sniper -> [Amazon, Coca, iPhone, Heineken, Kalash]
✅ Fi